# Machine Learning — Lab 9
## K-Nearest Neighbors and the Effect of Representation

**Main Course Learning Outcomes — CLO2, CLO4, CLO5**

- **CLO2:** Analyze datasets and apply appropriate preprocessing, transformation, and feature engineering techniques.
- **CLO4:** Apply supervised learning techniques to solve practical problems and interpret their outcomes.
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** KNN is simple to run but easy to misuse. Full credit requires you to explain **what “nearest” means, why scaling changes neighbors, how $k$ controls model flexibility, and how irrelevant features can damage distance-based learning**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. KNN mechanics | 20 min | Compute Euclidean distances and majority votes manually |
| 2. Unscaled KNN | 15 min | Observe how large-scale features dominate distance |
| 3. Scaled KNN | 25 min | Standardize features and compare neighbors/performance |
| 4. Choosing $k$ | 20 min | Study underfitting/overfitting across neighborhood sizes |
| 5. Representation experiment | 20 min | Add irrelevant features and measure the effect |
| 6. Decision boundary & interpretation | 10 min | Visualize how KNN partitions the feature space |
| 7. Debugging, challenge & viva | 10 min | Diagnose preprocessing and distance mistakes |
| **Total** | **120 min** | |

### Main idea

$$
\boxed{
\text{Representation}
\rightarrow
\text{Distance}
\rightarrow
\text{Neighbors}
\rightarrow
\text{Vote}
\rightarrow
\text{Prediction}
}
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. explain KNN as instance-based learning;
2. compute Euclidean distance manually;
3. identify the $k$ nearest training examples;
4. predict a class by majority vote;
5. explain why KNN has little conventional training but potentially expensive prediction;
6. demonstrate how feature scale changes nearest neighbors;
7. apply standardization correctly using training data only;
8. explain how small and large values of $k$ affect bias and variance;
9. evaluate the effect of irrelevant/noisy features on KNN;
10. select a practical KNN configuration using validation evidence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

print("Machine Learning Lab 9 environment ready.")

# Part I — KNN from First Principles

K-Nearest Neighbors does not learn a compact equation like linear regression.

Instead, it stores the training examples.

For a new point $x$:

1. compute its distance to training examples;
2. find the $k$ nearest;
3. collect their labels;
4. predict by majority vote.

For numerical features, Euclidean distance is:

$$
d(x,z)
=
\sqrt{
\sum_{j=1}^{p}
(x_j-z_j)^2
}.
$$

## Task 1.1 — Tiny Dataset

Suppose we want to classify a machine as:

- `0` = Normal
- `1` = Maintenance Needed

The training examples are:

In [ ]:
tiny_X = pd.DataFrame({
    "temperature": [50, 54, 60, 67, 72, 78],
    "vibration":   [1.0, 1.2, 1.6, 2.4, 2.8, 3.4],
})

tiny_y = np.array([0, 0, 0, 1, 1, 1])

display(
    tiny_X.assign(
        maintenance_needed=tiny_y
    )
)

new_point = np.array([65.0, 2.0])
print("New machine:", new_point)

## Task 1.2 — Implement Euclidean Distance

Complete:

In [ ]:
def euclidean_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    # TODO: implement Euclidean distance.
    distance = None

    return float(distance)

In [ ]:
assert abs(
    euclidean_distance(
        np.array([0.0, 0.0]),
        np.array([3.0, 4.0])
    ) - 5.0
) < 1e-12

print("Euclidean-distance test passed.")

## Task 1.3 — Manual Distances

Before running the next cell, calculate the distance from the new machine:

$$
(65,\;2.0)
$$

to at least **three** training examples by hand.

Show the squared differences and final square root.

Then predict the class for:

$$
k=3.
$$

**Your calculation:**

In [ ]:
distance_rows = []

for i, row in tiny_X.iterrows():
    d = euclidean_distance(
        row.to_numpy(),
        new_point
    )

    distance_rows.append({
        "row": i,
        "temperature": row["temperature"],
        "vibration": row["vibration"],
        "class": tiny_y[i],
        "distance": d,
    })

distance_table = pd.DataFrame(distance_rows).sort_values("distance")
display(distance_table.round(4))

## Task 1.4 — Implement Majority Vote

Complete the function.

For this binary lab, if a tie occurs, return the class of the **closest** neighbor in the supplied ordered list.

In [ ]:
def majority_vote(ordered_labels):
    labels = list(ordered_labels)

    counts = {
        0: labels.count(0),
        1: labels.count(1),
    }

    # TODO:
    # return majority class;
    # if tied, return the closest neighbor's class.
    prediction = None

    return int(prediction)

In [ ]:
assert majority_vote([1, 1, 0]) == 1
assert majority_vote([0, 1, 0]) == 0
assert majority_vote([1, 0]) == 1

print("Majority-vote tests passed.")

In [ ]:
for k in [1, 3, 5]:
    nearest_labels = (
        distance_table
        .head(k)["class"]
        .astype(int)
        .tolist()
    )

    print(
        f"k={k}: "
        f"labels={nearest_labels}, "
        f"prediction={majority_vote(nearest_labels)}"
    )

## Task 1.5 — Interpret $k$

Answer:

1. Why is $k=1$ highly sensitive to one nearby observation?
2. Why can larger $k$ produce smoother decisions?
3. Why can very large $k$ ignore useful local structure?
4. For binary classification, why is an odd $k$ often convenient?

# Part II — A Realistic Equipment Dataset

We now simulate industrial equipment measurements.

Target:

```text
maintenance_needed
```

Features:

- `temperature_c`
- `vibration_mm_s`
- `pressure_kpa`
- `rpm`
- `operating_hours`

These features use very different numerical scales.

That makes this dataset ideal for studying why **representation matters in KNN**.

In [ ]:
rng = np.random.default_rng(3452)
n = 950

temperature_c = np.clip(rng.normal(64, 12, n), 25, 105)
vibration_mm_s = np.clip(rng.gamma(2.0, 0.7, n), 0.1, 6.0)
pressure_kpa = np.clip(rng.normal(310, 55, n), 150, 500)
rpm = np.clip(rng.normal(2400, 620, n), 700, 4200)
operating_hours = np.clip(rng.gamma(3.0, 1600, n), 100, 14000)

risk = (
    1.2 * (temperature_c > 78)
    + 1.7 * (vibration_mm_s > 2.1)
    + 1.0 * (pressure_kpa < 250)
    + 0.7 * (pressure_kpa > 390)
    + 1.0 * (operating_hours > 6500)
    + 0.6 * ((rpm > 3200) & (vibration_mm_s > 1.6))
    + rng.normal(0, 0.8, n)
)

risk_prob = 1 / (1 + np.exp(-(risk - 1.5)))
maintenance_needed = rng.binomial(1, risk_prob)

df_master = pd.DataFrame({
    "temperature_c": np.round(temperature_c, 2),
    "vibration_mm_s": np.round(vibration_mm_s, 3),
    "pressure_kpa": np.round(pressure_kpa, 2),
    "rpm": np.round(rpm, 1),
    "operating_hours": np.round(operating_hours, 1),
    "maintenance_needed": maintenance_needed,
})

print("Master dataset shape:", df_master.shape)
print("\nClass balance:")
display(
    df_master["maintenance_needed"]
    .value_counts()
    .rename(index={0:"normal", 1:"maintenance_needed"})
)

## Task 2.1 — Personalized Working Dataset

Enter the last four digits of your student ID.

Your ID determines a reproducible sample of 760 machines.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 9000 + (STUDENT_ID_LAST4 % 1000)

df = df_master.sample(
    n=760,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your KNN seed:", SEED)
print("Working dataset shape:", df.shape)

In [ ]:
feature_cols = [
    "temperature_c",
    "vibration_mm_s",
    "pressure_kpa",
    "rpm",
    "operating_hours",
]

X = df[feature_cols].copy()
y = df["maintenance_needed"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=SEED,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

## Task 2.2 — Predict Which Features Dominate Distance

Inspect the ranges below.

Before running KNN, predict which feature is most likely to dominate raw Euclidean distance.

In [ ]:
range_table = pd.DataFrame({
    "min": X_train.min(),
    "max": X_train.max(),
    "range": X_train.max() - X_train.min(),
    "std": X_train.std(),
})

display(range_table.round(3))

### Questions

1. Which feature has the largest numerical range?
2. Which has the smallest?
3. Why does numerical range affect Euclidean distance?
4. Does a larger numerical range necessarily mean the feature is more important for maintenance?

# Part III — KNN Without Scaling

We first fit KNN directly on raw features.

This is intentionally risky.

In [ ]:
knn_raw = KNeighborsClassifier(
    n_neighbors=7
)

knn_raw.fit(X_train, y_train)

raw_train_pred = knn_raw.predict(X_train)
raw_valid_pred = knn_raw.predict(X_valid)

raw_results = {
    "train_accuracy": accuracy_score(y_train, raw_train_pred),
    "valid_accuracy": accuracy_score(y_valid, raw_valid_pred),
    "valid_f1": f1_score(y_valid, raw_valid_pred),
}

print("Unscaled KNN results:")
display(pd.Series(raw_results).round(4).to_frame("value"))

## Task 3.1 — Interpret Unscaled KNN

Answer:

1. Is the model performing well enough to trust immediately?
2. Which features probably dominate its distance calculations?
3. Why can the model ignore useful low-scale features such as vibration?
4. Why is this a representation problem rather than a KNN coding problem?

# Part IV — Standardize the Representation

Standardization uses:

$$
z=\frac{x-\mu}{\sigma}.
$$

After standardization, features are placed on comparable scales.

The scaler must be fitted on **training data only**.

In [ ]:
knn_scaled = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=7
    )),
])

knn_scaled.fit(X_train, y_train)

scaled_train_pred = knn_scaled.predict(X_train)
scaled_valid_pred = knn_scaled.predict(X_valid)

scaled_results = {
    "train_accuracy": accuracy_score(y_train, scaled_train_pred),
    "valid_accuracy": accuracy_score(y_valid, scaled_valid_pred),
    "valid_f1": f1_score(y_valid, scaled_valid_pred),
}

comparison = pd.DataFrame({
    "Unscaled KNN": raw_results,
    "Scaled KNN": scaled_results,
}).T

display(comparison.round(4))

## Task 4.1 — Compare Scaled and Unscaled KNN

Answer:

1. Did validation accuracy change?
2. Did validation $F_1$ change?
3. Why can scaling improve KNN even though no label information is used in the scaler?
4. Why does scaling change the **geometry** of the feature space?
5. Does scaling guarantee improvement in every dataset?

# Part V — How Scaling Changes the Actual Neighbors

We will choose one validation machine and inspect its nearest training examples:

- once using raw features;
- once using standardized features.

This makes the effect of representation concrete.

In [ ]:
sample_position = (SEED * 3) % len(X_valid)
sample_index = X_valid.index[sample_position]
sample_row = X_valid.loc[[sample_index]]

print("Selected validation machine:")
display(sample_row)
print("True class:", int(y_valid.loc[sample_index]))

## Task 5.1 — Predict Before Inspecting Neighbors

Before running the next cells:

1. Do you expect the raw-space and scaled-space nearest neighbors to be identical?
2. Which raw feature is likely to dominate the unscaled neighbors?
3. Which low-scale feature may become more influential after scaling?

In [ ]:
raw_distances, raw_indices = knn_raw.kneighbors(
    sample_row,
    n_neighbors=7,
    return_distance=True
)

raw_neighbor_rows = X_train.iloc[raw_indices[0]].copy()
raw_neighbor_rows["class"] = y_train.iloc[raw_indices[0]].to_numpy()
raw_neighbor_rows["distance"] = raw_distances[0]

print("Nearest neighbors in RAW feature space:")
display(raw_neighbor_rows.round(4))

In [ ]:
scaler = knn_scaled.named_steps["scale"]
knn_only = knn_scaled.named_steps["knn"]

X_train_scaled = scaler.transform(X_train)
sample_scaled = scaler.transform(sample_row)

scaled_distances, scaled_indices = knn_only.kneighbors(
    sample_scaled,
    n_neighbors=7,
    return_distance=True
)

scaled_neighbor_rows = X_train.iloc[scaled_indices[0]].copy()
scaled_neighbor_rows["class"] = y_train.iloc[scaled_indices[0]].to_numpy()
scaled_neighbor_rows["distance"] = scaled_distances[0]

print("Nearest neighbors in SCALED feature space:")
display(scaled_neighbor_rows.round(4))

## Task 5.2 — Compare the Neighbor Sets

Answer:

1. How many neighbors are shared between the two sets?
2. Which examples changed?
3. Why did they change even though the original data are identical?
4. Which neighbor set appears more reasonable from a multi-feature perspective?
5. What does this demonstrate about the phrase “nearest neighbor”?

# Part VI — Choosing the Number of Neighbors

The value of $k$ controls flexibility.

### Small $k$

- very local;
- sensitive to noise;
- low bias;
- high variance.

### Large $k$

- smoother;
- less sensitive to individual points;
- higher bias;
- may underfit.

We will compare:

$$
k\in\{1,3,5,7,11,17,25,35\}.
$$

## Task 6.1 — Predict Before Running

Predict:

1. which $k$ will give the highest training accuracy;
2. which values are most likely to overfit;
3. which very large values may underfit;
4. whether validation performance should peak somewhere in the middle.

In [ ]:
k_values = [1, 3, 5, 7, 11, 17, 25, 35]
k_rows = []
k_models = {}

for k in k_values:
    model = Pipeline(steps=[
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=k
        )),
    ])

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)

    k_rows.append({
        "k": k,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "valid_accuracy": accuracy_score(y_valid, valid_pred),
        "valid_f1": f1_score(y_valid, valid_pred),
    })

    k_models[k] = model

k_table = pd.DataFrame(k_rows)
display(k_table.round(4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    k_table["k"],
    k_table["train_accuracy"],
    marker="o",
    label="Training accuracy"
)
plt.plot(
    k_table["k"],
    k_table["valid_accuracy"],
    marker="o",
    label="Validation accuracy"
)
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Accuracy")
plt.title("KNN Complexity: Training vs. Validation")
plt.legend()
plt.show()

## Task 6.2 — Analyze $k$

Answer:

1. Which $k$ gives the highest training accuracy?
2. Which $k$ gives the highest validation $F_1$?
3. Which settings show evidence of overfitting?
4. Which settings appear too smooth?
5. Why should $k$ be chosen using validation evidence rather than training accuracy?

# Part VII — Distance-Weighted KNN

Ordinary KNN gives each neighbor one vote.

Distance-weighted KNN gives closer neighbors more influence.

In scikit-learn:

```python
weights="distance"
```

This can help when the closest examples should matter more strongly.

In [ ]:
best_k = int(
    k_table.loc[
        k_table["valid_f1"].idxmax(),
        "k"
    ]
)

uniform_model = k_models[best_k]

distance_model = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=best_k,
        weights="distance"
    )),
])

distance_model.fit(X_train, y_train)

uniform_valid_pred = uniform_model.predict(X_valid)
distance_valid_pred = distance_model.predict(X_valid)

weighted_comparison = pd.DataFrame({
    "uniform_vote": {
        "accuracy": accuracy_score(y_valid, uniform_valid_pred),
        "f1": f1_score(y_valid, uniform_valid_pred),
    },
    "distance_vote": {
        "accuracy": accuracy_score(y_valid, distance_valid_pred),
        "f1": f1_score(y_valid, distance_valid_pred),
    },
}).T

display(weighted_comparison.round(4))

## Task 7.1 — Interpret Weighted Voting

Answer:

1. Which voting strategy performs better on your validation split?
2. Why might distance weighting help?
3. Why might it also make the model sensitive to a very close noisy point?
4. Should weighted voting be selected automatically? Explain.

# Part VIII — Representation Experiment: Add Irrelevant Features

Distance-based methods can suffer when irrelevant dimensions are added.

Your student ID determines how many random noise features will be added:

$$
2,\;4,\;6,\text{ or }8.
$$

These features contain no useful signal about maintenance.

In [ ]:
noise_options = [2, 4, 6, 8]
N_NOISE = noise_options[SEED % len(noise_options)]

print("Your assigned number of irrelevant noise features:", N_NOISE)

## Task 8.1 — Predict Before Adding Noise

Predict:

1. what will happen to pairwise distances;
2. whether validation performance is likely to improve or worsen;
3. why irrelevant dimensions can make “nearest” less meaningful;
4. whether scaling alone can solve the problem of irrelevant features.

In [ ]:
noise_rng = np.random.default_rng(SEED)

X_train_noise = X_train.copy()
X_valid_noise = X_valid.copy()

for j in range(N_NOISE):
    col = f"noise_{j+1}"

    X_train_noise[col] = noise_rng.normal(
        0, 1, size=len(X_train_noise)
    )

    X_valid_noise[col] = noise_rng.normal(
        0, 1, size=len(X_valid_noise)
    )

noise_model = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=best_k
    )),
])

noise_model.fit(X_train_noise, y_train)

noise_valid_pred = noise_model.predict(X_valid_noise)

noise_results = {
    "accuracy": accuracy_score(y_valid, noise_valid_pred),
    "f1": f1_score(y_valid, noise_valid_pred),
}

original_results = {
    "accuracy": accuracy_score(y_valid, uniform_valid_pred),
    "f1": f1_score(y_valid, uniform_valid_pred),
}

display(pd.DataFrame({
    "Original features": original_results,
    f"+ {N_NOISE} noise features": noise_results,
}).T.round(4))

## Task 8.2 — Analyze the Noise Experiment

Answer:

1. Did validation accuracy change?
2. Did validation $F_1$ change?
3. Was your prediction correct?
4. Why can irrelevant standardized features still damage KNN?
5. Which earlier CLO2 skill becomes important here?

> Hint: feature selection and representation quality.

# Part IX — Two-Dimensional Decision Boundary

To visualize KNN behavior, we will use two features:

- `temperature_c`
- `vibration_mm_s`

The pipeline still standardizes them before computing distance.

In [ ]:
two_features = [
    "temperature_c",
    "vibration_mm_s",
]

two_model = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=best_k
    )),
])

two_model.fit(
    X_train[two_features],
    y_train
)

x1_min, x1_max = X_train["temperature_c"].min(), X_train["temperature_c"].max()
x2_min, x2_max = X_train["vibration_mm_s"].min(), X_train["vibration_mm_s"].max()

x1_grid = np.linspace(x1_min, x1_max, 220)
x2_grid = np.linspace(x2_min, x2_max, 220)

xx1, xx2 = np.meshgrid(x1_grid, x2_grid)

grid = pd.DataFrame({
    "temperature_c": xx1.ravel(),
    "vibration_mm_s": xx2.ravel(),
})

grid_prob = two_model.predict_proba(grid)[:, 1].reshape(xx1.shape)

plt.figure(figsize=(8, 5))
plt.contour(
    xx1,
    xx2,
    grid_prob,
    levels=[0.5],
    linewidths=2
)
plt.scatter(
    X_train["temperature_c"],
    X_train["vibration_mm_s"],
    c=y_train,
    alpha=0.65
)
plt.xlabel("Temperature (°C)")
plt.ylabel("Vibration (mm/s)")
plt.title(f"KNN Decision Boundary — k={best_k}")
plt.show()

## Task 9.1 — Interpret the Boundary

Answer:

1. Is the boundary a straight line?
2. Why can KNN produce nonlinear boundaries without fitting a nonlinear equation?
3. How would $k=1$ likely change the boundary?
4. How would a much larger $k$ likely change it?
5. Why does this visualization only show part of the full five-feature model?

# Part X — Deliberate Debugging

Consider the following incorrect workflow:

```python
train_scaler = StandardScaler()
X_train_scaled = train_scaler.fit_transform(X_train)

valid_scaler = StandardScaler()
X_valid_scaled = valid_scaler.fit_transform(X_valid)
```

Then KNN is trained on `X_train_scaled` and evaluated on `X_valid_scaled`.

The code runs, but the geometry is inconsistent.

## Task 10.1 — Explain the Bug

Answer:

1. Why are training and validation now represented in different coordinate systems?
2. Which mean/std should define the validation transformation?
3. What is the correct workflow?
4. Why is this especially harmful for a distance-based model?

## Task 10.2 — Fix the Scaling Workflow

Complete:

In [ ]:
debug_scaler = StandardScaler()

# TODO: fit on training data only.
# debug_scaler.fit(...)

# TODO: transform both train and validation with the same scaler.
X_train_debug = None
X_valid_debug = None

In [ ]:
if X_train_debug is None or X_valid_debug is None:
    raise ValueError("Complete the scaling-debugging task.")

assert X_train_debug.shape == X_train.shape
assert X_valid_debug.shape == X_valid.shape

print("Scaling workflow shape checks passed.")

# Part XI — Personalized Neighbor Challenge

Your student ID assigns a validation machine and a value of $k$.

You must reason about its prediction before checking the result.

In [ ]:
personal_k_options = [3, 5, 9, 13]
PERSONAL_K = personal_k_options[
    SEED % len(personal_k_options)
]

challenge_position = (SEED * 7) % len(X_valid)
challenge_index = X_valid.index[challenge_position]
challenge_row = X_valid.loc[[challenge_index]]

print("Your assigned k:", PERSONAL_K)
print("Assigned validation machine:")
display(challenge_row)

## Task 11.1 — Predict Before Running

Using the feature values:

1. Do you expect `maintenance_needed = 0` or `1`?
2. Which two features most strongly influence your reasoning?
3. Will the prediction necessarily match the nearest single training example?
4. Why does your assigned $k$ matter?

In [ ]:
personal_model = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=PERSONAL_K
    )),
])

personal_model.fit(X_train, y_train)

challenge_pred = int(
    personal_model.predict(challenge_row)[0]
)

challenge_prob = float(
    personal_model.predict_proba(challenge_row)[0, 1]
)

challenge_true = int(
    y_valid.loc[challenge_index]
)

print("Predicted class:", challenge_pred)
print("Estimated probability for class 1:", round(challenge_prob, 4))
print("True class:", challenge_true)

## Task 11.2 — Explain the Personalized Prediction

Answer:

1. Was your prediction correct?
2. What class did the majority of the selected neighbors support?
3. Is the estimated probability in KNN a calibrated probability guarantee? Explain.
4. Would changing $k$ potentially change this prediction?
5. Why should one individual case not determine whether the model is good?

# Part XII — Validation-Based Final Selection

We will choose between:

- best scaled uniform-vote KNN;
- scaled distance-weighted KNN with the same $k$.

Use validation $F_1$ as the primary criterion.

If scores are nearly identical, prefer the simpler uniform vote unless you can justify otherwise.

In [ ]:
uniform_f1 = f1_score(y_valid, uniform_valid_pred)
distance_f1 = f1_score(y_valid, distance_valid_pred)

if distance_f1 > uniform_f1:
    final_model_name = f"Scaled distance-weighted KNN, k={best_k}"
    final_model = distance_model
else:
    final_model_name = f"Scaled uniform KNN, k={best_k}"
    final_model = uniform_model

print("Selected validation model:", final_model_name)
print("Uniform validation F1:", round(uniform_f1, 4))
print("Distance-weighted validation F1:", round(distance_f1, 4))

## Task 12.1 — Justify the Model Choice

Write 3–5 sentences including:

- selected $k$;
- whether uniform or distance weighting was selected;
- validation accuracy;
- validation $F_1$;
- why scaling is part of the final model;
- one limitation of KNN for large datasets.

# Part XIII — Final Test Evaluation

Only now evaluate the selected KNN configuration on the final test set.

In [ ]:
test_pred = final_model.predict(X_test)

test_cm = confusion_matrix(y_test, test_pred)

test_results = {
    "accuracy": accuracy_score(y_test, test_pred),
    "f1": f1_score(y_test, test_pred),
}

print("Final model:", final_model_name)

print("\nTest confusion matrix:")
display(pd.DataFrame(
    test_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

print("\nFinal test metrics:")
display(
    pd.Series(test_results)
    .round(4)
    .to_frame("value")
)

## Task 13.1 — Final Generalization Statement

Write 4–6 sentences including:

- the selected KNN configuration;
- validation $F_1$;
- test accuracy;
- test $F_1$;
- whether test performance is reasonably consistent with validation;
- why standardized preprocessing must remain attached to the model;
- one risk of irrelevant/high-dimensional features.

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. How does KNN classify a new observation?
2. Compute Euclidean distance between two 2D points.
3. Why does KNN usually require scaling?
4. Why can $k=1$ overfit?
5. Why can very large $k$ underfit?
6. Why can irrelevant features hurt KNN even after scaling?
7. What is the difference between uniform and distance-weighted voting?
8. Why must validation data use the training scaler?
9. For your personalized machine, explain how $k$ affected the prediction.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What did comparing actual neighbor sets before and after scaling teach you?
2. Why is KNN called instance-based or lazy learning?
3. Which value of $k$ showed the clearest overfitting behavior?
4. Why is feature representation unusually important for KNN?
5. What is one reason you might choose a decision tree instead of KNN?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] completed Euclidean-distance function;
- [ ] manual distance calculations;
- [ ] completed majority-vote function;
- [ ] your own student-ID-derived dataset;
- [ ] raw-feature range analysis;
- [ ] unscaled KNN evaluation;
- [ ] scaled KNN evaluation;
- [ ] raw-vs-scaled neighbor comparison;
- [ ] $k$ experiment and interpretation;
- [ ] uniform-vs-distance weighting comparison;
- [ ] personalized irrelevant-feature experiment;
- [ ] decision-boundary interpretation;
- [ ] corrected scaling workflow;
- [ ] personalized neighbor challenge;
- [ ] validation-based final model selection;
- [ ] final test evaluation;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\boxed{
\text{feature scale}
\rightarrow
\text{distance}
\rightarrow
\text{neighbor set}
\rightarrow
\text{vote}
\rightarrow
\text{generalization}
}
$$

# Lab 9 Summary

You should now be able to explain why KNN is simple in concept but highly dependent on representation.

### Euclidean distance

$$
d(x,z)
=
\sqrt{
\sum_j(x_j-z_j)^2
}.
$$

### Main lessons

- KNN predicts from nearby stored training examples.
- Feature scale changes the geometry of the space.
- Standardization should be fitted on training data only.
- Small $k$ can overfit.
- Large $k$ can underfit.
- Distance weighting changes how neighbor votes contribute.
- Irrelevant dimensions can make neighborhoods less meaningful.
- Validation should select $k$ and other KNN choices.
- The final test set should be used only after model selection.

**Next lab:** Neural Networks — Forward Pass and Backpropagation.